Libraries

In [ ]:
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter, ImageOps, ImageStat, ImageDraw
import cv2 # Wymagane dla transformacji perspektywy
import time # Do mierzenia czasu

Loading and preparing files

In [ ]:
PANEL_DIR = "panele"     # Katalog z panelami PNG (przezroczyste tło)
ROOF_DIR = "dachy"           # Katalog ze zdjęciami dachów (tła)
OUTPUT_BASE_DIR = "aug"

AUG_PER_COMPOSITE = 1         # Ilość generowanych obrazów na każdy plik panelu
CLASS_ID = 0                     # Indeks klasy YOLO (Panel fotowoltaiczny)

# --- USTAWIENIA REALIZMU ---
PERSPECTIVE_CHANCE = 0.9  # 90% szans na zniekształcenie perspektywy
PERSPECTIVE_STRENGTH = (0.1, 0.25) # Min/Max siła zniekształcenia

SCALE_RANGE = (0.05, 0.15) # Min/Max % szerokości tła, jaki zajmie panel
POSITION_MAX_Y = 0.60 # Gwarancja umieszczenia na dachu (górne 60% obrazu)

# --- Tworzenie katalogów ---
OUTPUT_IMG_DIR = os.path.join(OUTPUT_BASE_DIR, "obrazy")
OUTPUT_LABEL_DIR = os.path.join(OUTPUT_BASE_DIR, "etykiety")

os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_LABEL_DIR, exist_ok=True)

print(f"Katalogi wyjściowe gotowe w: {OUTPUT_BASE_DIR}")

Preztwarzanie samego panela - modyfikacje

In [257]:
def apply_perspective_transform(img, abs_boxes):
    """
    Stosuje losową transformację perspektywy (mapowanie 3D) używając OpenCV.
    Symuluje to widok panelu pod kątem, np. na spadzistym dachu.
    """
    if random.random() < 0.9: # 90% szans na zniekształcenie
        panel_w, panel_h = img.size
        
        # Źródłowe 4 rogi (oryginalny prostokąt)
        src_points = np.float32([[0, 0], [panel_w, 0], [panel_w, panel_h], [0, panel_h]])

        # Losowe przesunięcie punktów docelowych (zniekształcenie)
        max_shift = max(panel_w, panel_h) * random.uniform(0.1, 0.25)
        small_shift = max(panel_w, panel_h) * 0.05
        
        dst_points = np.float32([
            [random.uniform(0, max_shift), random.uniform(0, max_shift)], # Lewy górny
            [panel_w - random.uniform(0, max_shift), random.uniform(0, max_shift)], # Prawy górny
            [panel_w - random.uniform(0, small_shift), panel_h - random.uniform(0, small_shift)], # Prawy dolny
            [random.uniform(0, small_shift), panel_h - random.uniform(0, small_shift)]  # Lewy dolny
        ])
        
        # Obliczenie macierzy transformacji
        M = cv2.getPerspectiveTransform(src_points, dst_points)
        
        # Konwersja PIL -> CV2 (z kanałem Alpha)
        img_cv = np.array(img.convert('RGBA'))
        
        # Transformacja obrazu (wypełnienie przezroczystością)
        warped_img_cv = cv2.warpPerspective(img_cv, M, (panel_w, panel_h), borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0, 0))
        
        # Konwersja CV2 -> PIL
        warped_img = Image.fromarray(warped_img_cv, 'RGBA')

        # Transformacja Bounding Boxa (całego panelu)
        bbox_pts = np.float32([[[0, 0]], [[panel_w, 0]], [[panel_w, panel_h]], [[0, panel_h]]])
        
        # Transformacja punktów BBoxa
        warped_bbox_pts = cv2.perspectiveTransform(bbox_pts, M).squeeze()
        
        # Obliczenie nowego, prostokątnego Bounding Boxa, który obejmuje zniekształcone punkty
        x_min = np.min(warped_bbox_pts[:, 0])
        y_min = np.min(warped_bbox_pts[:, 1])
        x_max = np.max(warped_bbox_pts[:, 0])
        y_max = np.max(warped_bbox_pts[:, 1])

        new_abs_boxes = [[CLASS_ID, int(x_min), int(y_min), int(x_max), int(y_max)]]
        
        return warped_img, new_abs_boxes
    
    return img, abs_boxes

In [258]:
def apply_scaling(img, abs_boxes):
    
    width, height = img.size
    scale = random.uniform(0.2, 0.8)
    try:
        resample_filter = Image.Resampling.LANCZOS
    except AttributeError:
        resample_filter = Image.LANCZOS 
        
    img = img.resize((int(width * scale), int(height * scale)), resample_filter)
    scale_x = img.size[0] / width
    scale_y = img.size[1] / height
   
    abs_boxes = [
        [cls, int(x1 * scale_x), int(y1 * scale_y), int(x2 * scale_x), int(y2 * scale_y)]
        for cls, x1, y1, x2, y2 in abs_boxes
    ]
    return img, abs_boxes

In [259]:
def apply_rotation(img):
    angle = random.uniform(-30, 30)
    img = img.convert("RGBA")
    img = img.rotate(angle, expand=True)
    img = img.convert("RGB")
    return img

In [260]:

def apply_blur(img):
    if random.random() < 0.5:
        img = img.filter(ImageFilter.GaussianBlur(random.uniform(0, 2)))
    return img


In [261]:
def apply_brightness(img):
    if random.random() < 0.7:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.6, 1.4))
    return img

In [262]:
def apply_contrast(img):
    if random.random() < 0.7:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.6, 1.4))
    return img

In [263]:
def apply_color(img):
    if random.random() < 0.5:
        img = ImageEnhance.Color(img).enhance(random.uniform(0.6, 1.4))
    return img

In [264]:
def apply_crop(img, abs_boxes):
    if random.random() < 0.5:
        crop_x = random.randint(0, int(img.size[0] * 0.1))
        crop_y = random.randint(0, int(img.size[1] * 0.1))
        crop_w = img.size[0] - crop_x
        crop_h = img.size[1] - crop_y
        img = img.crop((crop_x, crop_y, crop_w, crop_h))
        abs_boxes = [
            [cls, x1 - crop_x, y1 - crop_y, x2 - crop_x, y2 - crop_y]
            for cls, x1, y1, x2, y2 in abs_boxes
        ]
    return img, abs_boxes

In [265]:
def apply_flip(img, abs_boxes):
    if random.random() < 0.5:
        img = ImageOps.mirror(img)
        w = img.size[0]
        abs_boxes = [
            [cls, w - x2, y1, w - x1, y2] for cls, x1, y1, x2, y2 in abs_boxes
        ]
    return img, abs_boxes

In [266]:
def apply_padding(img, abs_boxes):
    if random.random() < 0.5:
        pad = random.randint(10, 50)
        img = ImageOps.expand(img, border=pad, fill=(0, 0, 0))
        abs_boxes = [
            [cls, x1 + pad, y1 + pad, x2 + pad, y2 + pad] for cls, x1, y1, x2, y2 in abs_boxes
        ]
    return img, abs_boxes

In [267]:
def apply_noise(img):
    if random.random() < 0.5:
        np_img = np.array(img).astype(np.int16)
        noise = np.random.normal(0, 25, np_img.shape)
        np_img = np.clip(np_img + noise, 0, 255).astype(np.uint8)
        img = Image.fromarray(np_img)
    return img

In [268]:
def transform_image_and_boxes(img, abs_boxes):
    """Główna sekwencja augmentacji."""
    img, abs_boxes = apply_perspective_transform(img, abs_boxes)
    
    # KROK 2: Standardowe Augmentacje
    img, abs_boxes = apply_scaling(img, abs_boxes) 
    img = apply_rotation(img)
    img = apply_blur(img)
    img = apply_brightness(img)
    img = apply_contrast(img)
    img = apply_color(img)
    img = apply_noise(img)
    img, abs_boxes = apply_flip(img, abs_boxes)
    
    return img, abs_boxes

In [269]:
def get_bbox_for_image(img):
    w, h = img.size
    xc = 0.5
    yc = 0.5
    bw = 1.0
    bh = 1.0
    return [[CLASS_ID, xc, yc, bw, bh]]

def save_yolo_labels(label_path, boxes):
    with open(label_path, 'w') as f:
        for box in boxes:
            cls, xc, yc, w, h = box
            f.write(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

def denormalize_boxes(boxes, width, height):
    abs_boxes = []
    for cls, xc, yc, w, h in boxes:
        abs_boxes.append([
            cls,
            int((xc - w / 2) * width),
            int((yc - h / 2) * height),
            int((xc + w / 2) * width),
            int((yc + h / 2) * height)
        ])
    return abs_boxes

def normalize_boxes(boxes, width, height):
    norm_boxes = []
    for cls, x1, y1, x2, y2 in boxes:
        x1 = max(0, min(x1, width))
        y1 = max(0, min(y1, height))
        x2 = max(0, min(x2, width))
        y2 = max(0, min(y2, height))
        xc = (x1 + x2) / 2 / width
        yc = (y1 + y2) / 2 / height
        w = (x2 - x1) / width
        h = (y2 - y1) / height
        if w > 0 and h > 0:
            norm_boxes.append([cls, xc, yc, w, h])
    return norm_boxes

Panele na dachach

In [270]:
def match_color_and_lighting(panel_img, background_patch):
    """
    Dopasowuje średnią jasność i kolorystykę panelu do fragmentu tła (dachu),
    na który ma być nałożony. Zapobiega to efektowi "naklejki".
    """
    panel_rgb = panel_img.convert("RGB")
    
    # Statystyki tła (patch, czyli obszar dachu, gdzie będzie panel)
    background_stat = ImageStat.Stat(background_patch)
    
    # Statystyki panelu
    panel_stat = ImageStat.Stat(panel_rgb)
    
    bg_mean = np.array(background_stat.mean)
    panel_mean = np.array(panel_stat.mean)
    
    # Różnica w jasności (mean R, G, B)
    mean_diff = bg_mean - panel_mean
    
    # Konwersja panelu do NumPy (z kanałem Alpha)
    panel_np = np.array(panel_img).astype(np.int16)
    
    # Zastosowanie korekty tylko do kanałów RGB
    for i in range(3):
        # Mnożnik losowy zapewnia, że dopasowanie nie jest idealne (większy realizm)
        panel_np[:, :, i] = np.clip(panel_np[:, :, i] + mean_diff[i] * random.uniform(0.5, 1.5), 0, 255) 
        
    corrected_panel_np = panel_np.astype(np.uint8)
    
    return Image.fromarray(corrected_panel_np, 'RGBA')

In [271]:
def composite_panel_realistically(panel_img_aug, abs_boxes_aug, background_img):
    """
    Nakłada panel na tło, kontrolując skalę, pozycję i dopasowanie kolorów.
    GWARANTUJE, że panel znajdzie się na dachu.
    """
    bg_w, bg_h = background_img.size
    panel_w, panel_h = panel_img_aug.size
    
    # 1. KONTROLA SKALI (Realistyczne Rozmiary na dachu)
    # Panel ma zająć 5% do 15% szerokości tła (symulacja widoku z drona/satelity).
    target_fill_percent = random.uniform(0.05, 0.15) 
    
    target_panel_w = int(bg_w * target_fill_percent)
    scale_needed = target_panel_w / panel_w
    
    new_w = target_panel_w
    new_h = int(panel_h * scale_needed)
    
    # Zastosowanie finalnego skalowania obrazu
    panel_img_aug = panel_img_aug.resize((new_w, new_h), Image.Resampling.LANCZOS)
    panel_w, panel_h = new_w, new_h
    
    # Skalowanie Bounding Boxa po skalowaniu kompozycyjnym
    abs_boxes_scaled = []
    for cls, x1, y1, x2, y2 in abs_boxes_aug:
        abs_boxes_scaled.append([
            cls,
            int(x1 * scale_needed),
            int(y1 * scale_needed),
            int(x2 * scale_needed),
            int(y2 * scale_needed)
        ])
    
    # 2. KONTROLA POZYCJI (Gwarancja Umieszczenia na Dachu)
    # Panel może trafić tylko do górnych 60% wysokości obrazu.
    # Zapobiega to umieszczeniu panelu na trawie, drodze lub ścianie bocznej.
    max_x = bg_w - panel_w
    max_y = int(bg_h * 0.60) - panel_h # Zacieśniamy do 60%

    if max_x <= 0 or max_y <= 0:
         return None, None # Pomijamy, jeśli tło jest za małe dla tej skali
    
    x_offset = random.randint(0, max_x)
    y_offset = random.randint(0, max_y) # Losowanie tylko w górnym obszarze

    # 3. DOPASOWANIE KOLORÓW (Wtopienie w Tło)
    # Wycięcie fragmentu tła w miejscu, gdzie trafi panel
    background_patch = background_img.crop((x_offset, y_offset, x_offset + panel_w, y_offset + panel_h))
    
    # Dopasowanie panelu do łatki tła
    panel_img_aug = match_color_and_lighting(panel_img_aug, background_patch)
    
    # 4. KOMPOZYCJA
    composite_img = background_img.copy()
    # Sklejenie, uwzględniając kanał alfa (przezroczystość)
    composite_img.paste(panel_img_aug, (x_offset, y_offset), panel_img_aug) 
    
    # 5. AKTUALIZACJA BBOXA (Dodanie przesunięcia)
    final_abs_boxes = []
    for cls, x1, y1, x2, y2 in abs_boxes_scaled:
        final_abs_boxes.append([
            cls, 
            x1 + x_offset, 
            y1 + y_offset, 
            x2 + x_offset, 
            y2 + y_offset
        ])
    
    return composite_img, final_abs_boxes

In [272]:
def get_initial_bbox(img):
    """Zwraca początkowy, znormalizowany BBox (pokrywa cały panel)"""
    return [[CLASS_ID, 0.5, 0.5, 1.0, 1.0]]

def denormalize_boxes(boxes, width, height):
    """Konwersja z formatu YOLO (xc, yc, w, h) na absolutne (x1, y1, x2, y2)"""
    abs_boxes = []
    for cls, xc, yc, w, h in boxes:
        abs_boxes.append([
            cls,
            int((xc - w / 2) * width),
            int((yc - h / 2) * height),
            int((xc + w / 2) * width),
            int((yc + w / 2) * height)
        ])
    return abs_boxes

def normalize_boxes(boxes, width, height):
    """Konwersja z absolutnych (x1, y1, x2, y2) na format YOLO (xc, yc, w, h)"""
    norm_boxes = []
    for cls, x1, y1, x2, y2 in boxes:
        # Przycinanie do granic obrazu
        x1 = max(0, min(x1, width))
        y1 = max(0, min(y1, height))
        x2 = max(0, min(x2, width))
        y2 = max(0, min(y2, height))
        
        xc = (x1 + x2) / 2 / width
        yc = (y1 + y2) / 2 / height
        w = (x2 - x1) / width
        h = (y2 - y1) / height
        
        # Filtrujemy, aby uniknąć pustych bounding boxów
        if w > 0 and h > 0:
            norm_boxes.append([cls, xc, yc, w, h])
    return norm_boxes

def save_yolo_labels(label_path, boxes):
    """Zapisuje znormalizowane boxy do pliku .txt w formacie YOLO"""
    with open(label_path, 'w') as f:
        for box in boxes:
            cls, xc, yc, w, h = box
            f.write(f"{int(cls)} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

Pętla główna

In [273]:
# for file in os.listdir(PANEL_DIR):
#     if not file.lower().endswith(('.jpg', '.png')):
#         continue

#     img_path = os.path.join(PANEL_DIR, file)
#     original_img = Image.open(img_path)
#     boxes = get_bbox_for_image(original_img)

#     for i in range(AUG_PER_IMAGE):
#         img = original_img.copy()
#         abs_boxes = denormalize_boxes(boxes, img.width, img.height)
#         aug_img, aug_boxes = transform_image_and_boxes(img, abs_boxes)
#         norm_boxes = normalize_boxes(aug_boxes, aug_img.width, aug_img.height)

#         out_name = f"{os.path.splitext(file)[0]}_aug_{i:03d}"
#         aug_img.save(os.path.join(OUTPUT_IMG_DIR, out_name + ".jpg"))
#         save_yolo_labels(os.path.join(OUTPUT_LABEL_DIR, out_name + ".txt"), norm_boxes)

#     print(f"✔ {file} - {AUG_PER_IMAGE} augmentacji")

In [274]:
print("Rozpoczynam generowanie danych...")

# 1. Wczytanie listy plików tła (dachów)
roof_files = [f for f in os.listdir(ROOF_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
if not roof_files:
    print(f"BŁĄD KRYTYCZNY: Brak plików w katalogu dachów: {ROOF_DIR}. Przerwij i dodaj obrazy tła.")
else:
    print(f"Znaleziono {len(roof_files)} obrazów tła.")

# 2. Pętla po plikach paneli
panel_files = [f for f in os.listdir(PANEL_DIR) if f.lower().endswith('.png')]
if not panel_files:
    print(f"BŁĄD KRYTYCZNY: Brak plików .png w katalogu paneli: {PANEL_DIR}. Przerwij i dodaj wycięte panele.")
else:
    print(f"Znaleziono {len(panel_files)} plików paneli. Rozpoczynam pętlę główną...")

total_generated_count = 0
for panel_file in panel_files:
    if not roof_files: # Sprawdzenie, czy mamy tła
        break
        
    print(f"\n--- Przetwarzanie panelu: {panel_file} ---")
    
    panel_path = os.path.join(PANEL_DIR, panel_file)
    original_panel_img = Image.open(panel_path).convert("RGBA") 
    initial_boxes_norm = get_initial_bbox(original_panel_img) 
    base_name = os.path.splitext(panel_file)[0]
    
    generated_count = 0
    
    # Ta pętla będzie działać, o ile zewnętrzne pętle znajdą pliki
    while generated_count < AUG_PER_IMAGE:
        try:
            # 1. Wczytanie tła
            roof_file = random.choice(roof_files)
            roof_path = os.path.join(ROOF_DIR, roof_file)
            roof_img = Image.open(roof_path).convert("RGB")
            
            # 2. Augmentacja panelu (Perspektywa, Skalowanie, Kolor, Szum)
            panel_copy = original_panel_img.copy()
            abs_boxes = denormalize_boxes(initial_boxes_norm, panel_copy.width, panel_copy.height)
            aug_panel, aug_abs_boxes = transform_image_and_boxes(panel_copy, abs_boxes)
            
            # 3. Realistyczna Kompozycja (Skala, Pozycja, Dopasowanie Kolorów)
            composite_img, final_abs_boxes = composite_panel_realistically(
                aug_panel, 
                aug_abs_boxes, 
                roof_img
            )
            
            if composite_img is None:
                continue # Pominięcie (np. tło za małe)

            # 4. Normalizacja i zapis
            norm_boxes = normalize_boxes(final_abs_boxes, composite_img.width, composite_img.height)

            if not norm_boxes:
                continue # Pominięcie (np. BBox poza kadrem)
                
            out_name = f"{base_name}_roof_{generated_count:03d}"
            composite_img.save(os.path.join(OUTPUT_IMG_DIR, out_name + ".jpg"))
            save_yolo_labels(os.path.join(OUTPUT_LABEL_DIR, out_name + ".txt"), norm_boxes)
            
            generated_count += 1
            total_generated_count += 1
            
            # Print postępu co 10 obrazów
            if generated_count % 10 == 0:
                print(f"    ...wygenerowano {generated_count}/{AUG_PER_IMAGE} obrazów dla {panel_file}")

        except Exception as e:
            # Wypisujemy błąd, ale kontynuujemy pętlę
            print(f"BŁĄD podczas generowania dla {panel_file} (tło: {roof_file}): {e}") 
            continue
  
    print(f"✔ Panel {panel_file}: Zakończono. Zapisano {generated_count} obrazów.")
        
print(f"\n✅ Generowanie zakończone. Zbiór danych znajduje się w '{OUTPUT_BASE_DIR}'.")

Rozpoczynam generowanie danych...
Znaleziono 38 obrazów tła.
Znaleziono 5 plików paneli. Rozpoczynam pętlę główną...

--- Przetwarzanie panelu: 1402181-small-removebg-preview.png ---
BŁĄD podczas generowania dla 1402181-small-removebg-preview.png (tło: Screenshot 2025-05-05 165453.png): buffer is not large enough
BŁĄD podczas generowania dla 1402181-small-removebg-preview.png (tło: Screenshot 2025-05-05 143359.png): buffer is not large enough
BŁĄD podczas generowania dla 1402181-small-removebg-preview.png (tło: Screenshot 2025-05-05 165523.png): buffer is not large enough
BŁĄD podczas generowania dla 1402181-small-removebg-preview.png (tło: Screenshot 2025-05-05 165712.png): buffer is not large enough
BŁĄD podczas generowania dla 1402181-small-removebg-preview.png (tło: Screenshot 2025-10-20 210005.png): buffer is not large enough
BŁĄD podczas generowania dla 1402181-small-removebg-preview.png (tło: Screenshot 2025-05-05 165655.png): buffer is not large enough
BŁĄD podczas generowania 

KeyboardInterrupt: 